In [1]:
import xgboost as xgb
from sklearn.metrics import accuracy_score
import time

In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/MedAssist AI/clean_190k_dataset.csv')

In [3]:
from sklearn.preprocessing import LabelEncoder
df_clean = df.copy()
# STEP 1: LABEL ENCODING
print("Translating diseases to numbers...")
encoder = LabelEncoder()
df_clean['target'] = encoder.fit_transform(df_clean['diseases'])
df_final = df_clean.drop(columns=['diseases'])

Translating diseases to numbers...


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("Isolating rare diseases to protect them from the split...")

# 1. Figure out which diseases are rare (1 row) and which are common (2+ rows)
class_counts = df_final['target'].value_counts()
rare_classes = class_counts[class_counts == 1].index
common_classes = class_counts[class_counts > 1].index

# 2. Split the dataset into two separate dataframes
df_rare = df_final[df_final['target'].isin(rare_classes)]
df_common = df_final[df_final['target'].isin(common_classes)]

# 3. Perform the Stratified Split ONLY on the common diseases
X_common = df_common.drop(columns=['target'])
y_common = df_common['target']

X_train_common, X_test, y_train_common, y_test = train_test_split(
    X_common, y_common, test_size=0.2, random_state=42, stratify=y_common
)

# 4. Manually force all the rare diseases directly into the Training Set
X_rare = df_rare.drop(columns=['target'])
y_rare = df_rare['target']

X_train = pd.concat([X_train_common, X_rare])
y_train = pd.concat([y_train_common, y_rare])

print("--- DATA PRESERVATION COMPLETE ---")
print(f"Total diseases preserved: {len(y_train.unique())}")
print(f"Training on {X_train.shape[0]} rows...")
print(f"Testing on {X_test.shape[0]} rows...")

Isolating rare diseases to protect them from the split...
--- DATA PRESERVATION COMPLETE ---
Total diseases preserved: 773
Training on 151726 rows...
Testing on 37921 rows...


In [5]:
print("Initializing XGBoost...")
start_time = time.time()
# Count the exact number of unique diseases in your dataset
total_diseases = len(df_final['target'].unique())

Initializing XGBoost...


In [7]:
xgb_model = xgb.XGBClassifier(
    tree_method='hist',
    random_state=42,
    num_class=total_diseases,
    objective='multi:softprob', # Forces multi-class probability math
    n_jobs=-1,          # Forces the use of all CPU cores
    n_estimators=50
)

In [9]:
print("Training XGBoost...")
# 2. Define the evaluation set (so it knows what to test against during training)
eval_set = [(X_train, y_train), (X_test, y_test)]

# 3. Add eval_set and verbose to the fit command
xgb_model.fit(
    X_train, y_train,
    eval_set=eval_set,
    verbose=1 # This tells it to print an update after every single tree (1 by 1)
)

Training XGBoost...
[0]	validation_0-mlogloss:2.89882	validation_1-mlogloss:3.04922
[1]	validation_0-mlogloss:2.24480	validation_1-mlogloss:2.37263
[2]	validation_0-mlogloss:1.84173	validation_1-mlogloss:1.99746
[3]	validation_0-mlogloss:1.49922	validation_1-mlogloss:1.65038
[4]	validation_0-mlogloss:1.22539	validation_1-mlogloss:1.37582
[5]	validation_0-mlogloss:1.06010	validation_1-mlogloss:1.21548
[6]	validation_0-mlogloss:0.92094	validation_1-mlogloss:1.08417
[7]	validation_0-mlogloss:0.81396	validation_1-mlogloss:0.97398
[8]	validation_0-mlogloss:0.74520	validation_1-mlogloss:0.90524
[9]	validation_0-mlogloss:0.69204	validation_1-mlogloss:0.85233
[10]	validation_0-mlogloss:0.64994	validation_1-mlogloss:0.81097
[11]	validation_0-mlogloss:0.61411	validation_1-mlogloss:0.77653
[12]	validation_0-mlogloss:0.58211	validation_1-mlogloss:0.74584
[13]	validation_0-mlogloss:0.55461	validation_1-mlogloss:0.72039
[14]	validation_0-mlogloss:0.53122	validation_1-mlogloss:0.69867
[15]	validation

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=-1, num_class=773, ...)

### Why is the model training taking so long?

The training duration for an XGBoost model, especially in a multi-class classification problem, is influenced by several key factors:

1.  **Dataset Size:** Your training dataset (`X_train`) has `151,726` rows and `377` columns. Larger datasets naturally require more computation.
2.  **Number of Classes:** You have `773` unique diseases (`num_class=773`). Multi-class classification problems with a high number of classes are significantly more computationally intensive than binary classification, as the model essentially needs to differentiate between many more outcomes.
3.  **Number of Estimators (`n_estimators`):** You've set `n_estimators=50`. This means the model is building 50 individual decision trees. While 50 is a moderate number, each tree's construction involves complex calculations over the entire dataset.
4.  **Tree Complexity (implicitly):** The `tree_method='hist'` is generally faster than the default `exact` method for large datasets, but the underlying complexity of fitting each tree to differentiate between 773 classes still contributes to the time.
5.  **Hardware:** Although you're using `n_jobs=-1` to utilize all available CPU cores, a CPU-only environment might still be slower compared to GPU-accelerated training for large models.

### Potential Solutions to Speed Up Training:

*   **Reduce `n_estimators` for Prototyping:** For initial experimentation and faster feedback, you could temporarily reduce `n_estimators` to a smaller value (e.g., `10` or `20`). Once you have a good understanding of other parameters, you can increase it again.
*   **Enable GPU Acceleration:** If you're using Google Colab, consider changing your runtime type to include a GPU (Runtime -> Change runtime type -> Hardware accelerator -> GPU). XGBoost can leverage GPUs for significant speedups, especially with `tree_method='hist'`. You would then need to set `tree_method='gpu_hist'` in your `XGBClassifier` instantiation.
*   **Early Stopping:** While not directly reducing the `n_estimators` you set, you can use early stopping during training. This monitors a validation metric and stops training if the metric doesn't improve for a certain number of rounds (`early_stopping_rounds`). This can prevent overfitting and save time if the model converges early. You would need a separate validation set for this.

In [10]:
print("Making predictions and grading...")
# Passing the 20% final exam data
xgb_predictions = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_predictions)

Making predictions and grading...


In [11]:
end_time = time.time()

In [12]:
print("--- MODEL COMPLETE ---")
print(f"XGBoost Accuracy: {xgb_accuracy * 100:.2f}%")
print(f"Total Time Taken: {(end_time - start_time) / 60:.2f} minutes")

--- MODEL COMPLETE ---
XGBoost Accuracy: 78.75%
Total Time Taken: 69.37 minutes
